# Step 1: Centralized Baseline
**TSU NSF AI Workshop 2026 — Federated Learning Lab**

---

## What are we doing in this notebook?

Before we build a federated learning system, we need a **baseline** to compare against.

In this notebook we train a single AI model on **all the data at once** — the traditional (centralized) approach. This tells us the best accuracy we could achieve if we had no privacy constraints.

**Think of it this way:**
- Imagine all 3 hospitals send their patient images to one central computer.
- We train one model on everything.
- We record how accurate it is.
- Later, we will compare federated learning's accuracy to this number.

---

### Dataset: DermaMNIST
We are using **DermaMNIST** — a collection of skin lesion photographs.

| Property | Value |
|----------|-------|
| Image size | 28 × 28 pixels, color (RGB) |
| Training images | 7,007 |
| Test images | 2,005 |
| Classes | 7 types of skin lesions |

The AI's job: look at a photo and predict which of the 7 lesion types it is.

## 🔧 Setup: Mount Google Drive

Your data file (`dermamnist.npz`) should be stored in Google Drive at:
```
My Drive/
└── FederatedLearning/
    └── data/
        └── dermamnist.npz
```

**How to upload the file:**
1. Go to [drive.google.com](https://drive.google.com)
2. Create a folder called `FederatedLearning`, then inside it create a folder called `data`
3. Upload `dermamnist.npz` into that `data` folder

Run the cell below to connect Colab to your Google Drive. A permission popup will appear — click **Allow**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## 📁 Set File Paths

We point all our file operations to your Google Drive folder.
Results will also be saved there so you can access them after the session.

In [ ]:
import os

# Base folder in your Google Drive
DRIVE_BASE  = "/content/drive/MyDrive/FederatedLearning"
DATA_PATH   = os.path.join(DRIVE_BASE, "data", "dermamnist.npz")
RESULTS_DIR = os.path.join(DRIVE_BASE, "results")

# Create results folder if it does not exist
os.makedirs(RESULTS_DIR, exist_ok=True)

# Quick check: does the data file exist?
if os.path.exists(DATA_PATH):
    print(f"Data file found: {DATA_PATH}")
else:
    print("ERROR: Data file not found!")
    print(f"Expected location: {DATA_PATH}")
    print("Please upload dermamnist.npz to your Google Drive first.")

## 📦 Import Libraries

We use:
- **NumPy** — for loading and handling arrays of numbers
- **PyTorch** — for building and training the neural network
- **JSON** — for saving our results to a file

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json

# ---------- Settings ----------
EPOCHS        = 20    # How many times we loop through all training data
BATCH_SIZE    = 64    # How many images the model sees at once
LEARNING_RATE = 0.001 # How large each learning step is
NUM_CLASSES   = 7     # DermaMNIST has 7 skin lesion categories

# Use GPU if available (Colab offers free GPU — go to Runtime > Change runtime type > GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Load the Data

The `.npz` file contains the images and labels already split into train / validation / test sets.

- **Train set** — used to teach the model (7,007 images)
- **Validation set** — used to monitor performance during training (1,003 images)
- **Test set** — used for the final accuracy measurement (2,005 images)

In [ ]:
data = np.load(DATA_PATH)

train_images = data["train_images"]              # shape: (7007, 28, 28, 3)
val_images   = data["val_images"]                # shape: (1003, 28, 28, 3)
test_images  = data["test_images"]               # shape: (2005, 28, 28, 3)
train_labels = data["train_labels"].flatten()    # shape: (7007,)
val_labels   = data["val_labels"].flatten()      # shape: (1003,)
test_labels  = data["test_labels"].flatten()     # shape: (2005,)

print(f"Training images  : {train_images.shape}")
print(f"Validation images: {val_images.shape}")
print(f"Test images      : {test_images.shape}")

# Show class distribution
CLASS_NAMES = ["Actinic keratoses", "Basal cell carcinoma", "Benign keratosis",
               "Dermatofibroma", "Melanoma", "Melanocytic nevi", "Vascular lesions"]
print("\nClass distribution in training set:")
for c, name in enumerate(CLASS_NAMES):
    count = np.sum(train_labels == c)
    bar = "█" * (count // 30)
    print(f"  Class {c} ({name:<25}): {count:4d}  {bar}")

## Step 2: Prepare Data for PyTorch

Neural networks expect:
1. **Float values between 0 and 1** — so we divide pixel values (0–255) by 255
2. **Shape (N, C, H, W)** — channels first — so we rearrange from (N, H, W, C)

A `DataLoader` then feeds the data in small batches during training.

In [ ]:
def to_tensor_dataset(images, labels):
    x = torch.tensor(images / 255.0, dtype=torch.float32)
    x = x.permute(0, 3, 1, 2)   # (N,H,W,C) → (N,C,H,W)
    y = torch.tensor(labels, dtype=torch.long)
    return TensorDataset(x, y)

train_loader = DataLoader(to_tensor_dataset(train_images, train_labels),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(to_tensor_dataset(val_images, val_labels),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(to_tensor_dataset(test_images, test_labels),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches  : {len(train_loader)}  (each batch = {BATCH_SIZE} images)")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches      : {len(test_loader)}")

## Step 3: Define the Neural Network

We use a simple **Convolutional Neural Network (CNN)**.

```
Input image (3 × 28 × 28)
    ↓
Conv layer 1  →  32 filters  →  detects simple edges & colors
    ↓  MaxPool  →  14 × 14
Conv layer 2  →  64 filters  →  detects textures & shapes
    ↓  MaxPool  →  7 × 7
Flatten  →  3136 numbers
    ↓
Fully connected  →  128 neurons
    ↓
Output  →  7 scores  (one per lesion type)
```

The class with the **highest score** is the model's prediction.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 28×28 → 14×14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 14×14 → 7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SimpleCNN(num_classes=NUM_CLASSES).to(device)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created. Total parameters: {total_params:,}")

## Step 4: Train the Model

Each training epoch:
1. **Forward pass** — feed images through the network, get predictions
2. **Compute loss** — measure how wrong the predictions are (CrossEntropyLoss)
3. **Backward pass** — calculate how to adjust each weight to reduce the error
4. **Update weights** — take one step in the right direction (Adam optimizer)

We repeat this for every batch, every epoch.

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
history   = {"train_acc": [], "val_acc": []}

print("Training...")
print(f"{'Epoch':<8} {'Train Acc':>10} {'Val Acc':>10}")
print("-" * 30)

for epoch in range(1, EPOCHS + 1):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    train_acc = evaluate(model, train_loader, device)
    val_acc   = evaluate(model, val_loader,   device)
    history["train_acc"].append(round(train_acc, 4))
    history["val_acc"].append(round(val_acc, 4))
    print(f"{epoch:<8} {train_acc:>10.2%} {val_acc:>10.2%}")

## Step 5: Final Evaluation & Save Results

We evaluate on the **test set** — images the model has never seen — for the final accuracy score.
Results are saved to your Google Drive so the visualization notebook can use them.

In [ ]:
test_acc = evaluate(model, test_loader, device)
print(f"Final Test Accuracy: {test_acc:.2%}")

results = {
    "method"   : "Centralized",
    "epochs"   : EPOCHS,
    "test_acc" : round(test_acc, 4),
    "history"  : history,
}

results_path = os.path.join(RESULTS_DIR, "centralized_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {results_path}")
print("\n✅ Done! Open notebook 2_data_partitioning.ipynb next.")

## Summary

| What we did | Result |
|-------------|--------|
| Loaded all DermaMNIST training data | 7,007 images, 7 classes |
| Trained a CNN for 20 epochs | See accuracy above |
| Saved results | `results/centralized_results.json` |

**This is our benchmark.** In later notebooks, federated learning will try to match this accuracy — but without any client sharing their raw data.

➡️ **Next step: `2_data_partitioning.ipynb`** — split the data between 2 simulated hospitals.